# Session02: 大規模言語モデルの医学画像分析質問への対応精度評価

## Motivation（モチベーション）

この演習は、以下を学ぶことを目的としています：

- 医学画像に関する多肢選択問題を使って、モデルの実務的な性能を評価する方法を身につける。
- CSV データの前処理、集計、表・図の作成を通じて再現可能な解析ワークフローを学ぶ。
- 人間の受験者（student baseline）と LLM の結果を比較し、モデルの強み・弱みや分野依存性を理解する。
- コードを書き、可視化や統計指標の解釈を行うことで論文の Results および Materials and Methods セクションを構築する練習をする。

以下のノートブックは自分でコードを書きながら解析を進められるよう、実装のステップを記載しています。Google Colab用のリンクは後でスクリプトで差し替えてください。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session02_materials_and_results.ipynb)

## Colaboを開いた後の手順

data, imagesフォルダをそのままColaboのフォルダにコピーしてください。中身は手本の画像およびダミーデータです。自分で作成した表・図は、提出用として `./save_figures` フォルダに保存してください。保存するファイル名は `Table_1.png`, `Table_2.png`, `Figure_1.png`, `Figure_2.png` に統一します。

## 1) data の中身について (説明)

このノートブックでは、実際の 2024 年放射線診断学専門医試験の問題をもとに、LLM を API で呼び出して解かせた結果を、少し変更したダミーデータを扱います。データファイル: `markdowns/data/dummy_LLM.csv` を読み込んでください。

主なカラムは次の通りです:
- `Subspeciality`: 問題が属する医学専門分野 (例: Neuroradiology & Head & Neck)
- `Answer`: 正解ラベル (多肢選択の正解)
- `*_ans` カラム群: 各モデルが出力した候補回答 (例: `GPT-4.1_ans`)
- モデル名だけのカラム群: 各問題に対してそのモデルが正答したかどうかの二値値 (1=正解, 0=不正解)。カラム名は `GPT-4.1`, `o3`, `Claude 3.7 Sonnet`, など。

注意: 実データを用いるときは、`pd.read_csv` で読み込み、列名とデータ型を必ず確認してください。


In [ ]:
# データ読み込みの例
# pandas を使って CSV を読み込む: df = pd.read_csv('path/to/dummy_LLM.csv')
# カラム一覧を表示して、どのカラムが正解フラグか確認する: df.columns.tolist()
# 欠損値や余分な空白を取り除く: df = df.dropna() / df[col].str.strip()

## 2) 各分野の問題数を数えて Table をつくる

目標: 各 `Subspeciality` ごとの問題数を集計し、表（Table 1）を作成します。

次の手順を参考に進めてください。
- `df['Subspeciality'].value_counts()` で各分野の件数を取得します。
- 結果を `pd.DataFrame` に変換し、列名を `['Subspeciality', 'count']` のように整えます。
- 表を markdown に埋め込むには `df.to_markdown()` を使うか、表を画像として保存します。
- 作成した Table 1 は `./save_figures/Table_1.png` として保存します。
- ノートブック内で確認するときは、Table 1 のキャプションとともに `./save_figures/Table_1.png` を表示します。

In [ ]:
# 次のように実装できます:
# counts = df['Subspeciality'].value_counts().sort_index()
# table = counts.reset_index()
# table.columns = ['Subspeciality', 'count']

# import os
# os.makedirs('./save_figures', exist_ok=True)
# fig, ax = plt.subplots(figsize=(8, 3))
# ax.axis('off')
# ax.table(cellText=table.values, colLabels=table.columns, loc='center')
# plt.savefig('./save_figures/Table_1.png', dpi=300, bbox_inches='tight')

### Example: 質問数分布表の挿入例

以下は見本画像 `questions_table.png` を埋め込む例です。自分で作成した Table 1 は `./save_figures/Table_1.png` として保存してください。
 
![質問数分布表](./images/questions_table.png)

## 3) accuracy を求める

目標: 各モデルの正解率（accuracy）を定義し、全体と分野別に計算して Table 2 を作成します。

次の手順を参考に進めてください。
- 各モデルの正解フラグカラム（例: `GPT-4.1`）は 1/0 で与えられています。モデルごとの accuracy は `df[model].mean()` で求められます。
- パーセンテージ表示にする場合は `*100` を使います。
- 分野別 accuracy は `df.groupby('Subspeciality')[model].mean()` で求めます。
- 表の形式は、行をモデル、列を全体と各分野としたクロス表に整えます。
- 作成した Table 2 は `./save_figures/Table_2.png` として保存します。

In [ ]:
# 次のように実装できます:
# models = ['GPT-4.1','o3', 'Claude 3.7 Sonnet', ...]
# overall = {m: df[m].mean() * 100 for m in models}
# by_field = df.groupby('Subspeciality')[models].mean() * 100
# accuracy_table = by_field.T
# accuracy_table.insert(0, 'Overall', pd.Series(overall))
# accuracy_table = accuracy_table.round(1).reset_index().rename(columns={'index': 'Model'})
# import os
# os.makedirs('./save_figures', exist_ok=True)
# fig, ax = plt.subplots(figsize=(10, 4))
# ax.axis('off')
# ax.table(cellText=accuracy_table.values, colLabels=accuracy_table.columns, loc='center')
# plt.savefig('./save_figures/Table_2.png', dpi=300, bbox_inches='tight')

### Example: 精度テーブルの挿入例

以下は見本画像 `accuracy_table.png` を埋め込む例です。自分で作成した Table 2 は `./save_figures/Table_2.png` として保存してください。
 
![精度テーブル](./images/accuracy_table.png)

## 4) 受験生の平均、SD を求め、LLMの正解率と比較してプロットする

目標: 受験生（Human）がいると仮定した場合の平均得点と標準偏差を求め、それを LLM の精度と同じ図に重ねて比較します。

次の手順を参考に進めてください。
- 受験生データがあればその CSV を読み込み、各受験生の正答数を問題数で割って得点率を計算します。
- 受験生データがない場合は、`df['Answer']` を基準に多数決正答を human baseline として模擬する方法も考えられます。
- 平均と SD は `students_scores.mean()` / `students_scores.std()` で求めます。
- プロットでは、LLM のバープロットに `plt.axhline(student_mean, color='k', linestyle='--')` を重ね、必要に応じて `plt.fill_between(x, student_mean-student_sd, student_mean+student_sd, alpha=0.2)` を使います。
- 作成した Figure 1 は `./save_figures/Figure_1.png` として保存します。

In [ ]:
# 次のように実装できます:
# student_scores = pd.read_csv('students_scores.csv')
# student_scores['score'] = student_scores['correct_count'] / total_questions * 100
# mean = student_scores['score'].mean()
# sd = student_scores['score'].std()
# sns.barplot(x=list(overall.keys()), y=list(overall.values()))
# plt.axhline(mean, color='k', linestyle='--')
# plt.fill_between(range(len(overall)), mean-sd, mean+sd, alpha=0.2)
# import os
# os.makedirs('./save_figures', exist_ok=True)
# plt.savefig('./save_figures/Figure_1.png', dpi=300, bbox_inches='tight')

### Example: 受験生平均とLLMの比較（挿入例）

以下は受験生平均や標準偏差をプロットした見本画像です。自分で作成した Figure 1 は `./save_figures/Figure_1.png` として保存してください。
 
![精度分布グラフ](./images/model_performance_distribution.png)

## 5) 各分野のLLM正解率をプロットしてレーダーチャートをつくる

目標: 各 `Subspeciality` ごとにモデルの正解率を集計し、モデルごとにレーダーチャートで比較します。

次の手順を参考に進めてください。
- `by_field = df.groupby('Subspeciality')[models].mean()` で分野別正答率を求めます。
- レーダーチャートでは各分野を角度にマッピングし、最後の点を先頭に追加してループを閉じます。
- matplotlib の `projection='polar'` を使って複数モデルを重ねて描画します。
- 軸のレンジを 0-100 (%) に揃えると見やすくなります。
- 作成した Figure 2 は `./save_figures/Figure_2.png` として保存します。

In [ ]:
# 次のように実装できます:
# categories = list(by_field.index)
# values = by_field.loc['GPT-4.1'].tolist()  # 例
# values += values[:1]
# angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
# angles += angles[:1]
# ax = plt.subplot(projection='polar')
# ax.plot(angles, values, label='GPT-4.1')
# ax.fill(angles, values, alpha=0.25)
# import os
# os.makedirs('./save_figures', exist_ok=True)
# plt.savefig('./save_figures/Figure_2.png', dpi=300, bbox_inches='tight')

### Example: レーダーチャートの挿入例

以下は分野ごとのモデル正解率を比較したレーダーチャートの見本画像です。自分で作成した Figure 2 は `./save_figures/Figure_2.png` として保存してください。
 
![レーダーチャート](./images/radar_chart.png)

## 6) Markdown で表・図へリンクする

この実習では、作成した表・図を `./save_figures` に保存し、論文ドラフトの Markdown からリンクします。Markdown では、画像を表示したい場所に次の形式で書きます。

```markdown
![表示名](画像ファイルへの相対パス)
```

このノートブックと同じ場所に `save_figures` フォルダがある場合、以下のように書きます。ファイル名には空白を使わず、`_` を使うと Pandoc で PDF に変換するときも安全です。

```markdown
![Table 1](./save_figures/Table_1.png)
![Table 2](./save_figures/Table_2.png)
![Figure 1](./save_figures/Figure_1.png)
![Figure 2](./save_figures/Figure_2.png)
```

論文ドラフトでは、画像リンクの直前または直後に Table Legend / Figure Legend を書きます。例:

```markdown
Table 1. Distribution of radiology board examination questions by subspeciality.

![Table 1](./save_figures/Table_1.png)

Figure 1. Comparison of LLM accuracy with examinee performance.

![Figure 1](./save_figures/Figure_1.png)
```

注意: `./images` は見本画像を見るためのフォルダです。自分で作成した提出用の表・図は `./save_figures` からリンクしてください。

### Pandoc での図表参照について

通常の Pandoc だけで PDF に変換する場合、画像は埋め込まれますが、`Figure 1` や `Table 1` の番号を自動で相互参照する機能は限定的です。そのため、この実習では本文中で `Table 1`, `Table 2`, `Figure 1`, `Figure 2` と手動で書いて参照します。

```markdown
The distribution of questions is shown in Table 1.

Table 1. Distribution of radiology board examination questions by subspeciality.

![Table 1](./save_figures/Table_1.png)
```

より本格的に自動番号付け・相互参照を行う場合は、`pandoc-crossref` という Pandoc filter を使います。その場合は、図表に ID を付け、本文中で `@fig:...` や `@tbl:...` と引用します。

```markdown
The comparison with examinee performance is shown in Figure @fig:model_vs_examinees.

![Comparison of LLM accuracy with examinee performance.](./save_figures/Figure_1.png){#fig:model_vs_examinees}
```

表を画像として扱う場合も、次のように `fig:` ID を付けられます。表として厳密に扱う場合は Markdown table に `tbl:` ID を付けます。

```markdown
The overall and subspeciality-specific accuracies are shown in Table @fig:accuracy_table.

![Overall and subspeciality-specific accuracy of LLMs.](./save_figures/Table_2.png){#fig:accuracy_table}
```

`pandoc-crossref` を使う場合の変換例:

```bash
pandoc my_paper.md --filter pandoc-crossref -o my_paper.pdf
```

このフィルタがない環境では、`@fig:...` や `@tbl:...` は正しく処理されません。授業では、まず手動で `Table 1` / `Figure 1` と書く方法を使うのが安全です。


## 7) Table Legend を書く

### Table Legend とは

Table Legend（表の説明文、caption）は、読者が本文を読まなくても「この表が何を示しているか」を理解できるようにする短い説明です。表そのものに書ききれない補足情報を整理して書きます。

Table Legend に含める内容:
- 表番号と短いタイトル: `Table 1. Distribution of questions by subspeciality.` のように書きます。
- 対象と単位: 何を数えた表か、`n`、`%`、accuracy などの単位を明記します。
- 略語の説明: LLM、SD など、表だけを見た読者に必要な略語を定義します。
- 計算方法の補足: accuracy を「正解数 / 問題数」として計算した、などを簡潔に書きます。

注意点:
- Legend は結果の解釈を長く述べる場所ではありません。
- 「どのモデルが最も優れていた」などの主張は Results 本文に書きます。
- 表中のすべての略語は、本文で説明済みでも legend 内で再度説明するのが基本です。


### Exercise 7-1: Table 1 の Legend を書く

Table 1 は、各 `Subspeciality` に含まれる問題数を示す表です。次の空欄を自分のデータに合わせて埋めてください。

```markdown
Table 1. Distribution of radiology board examination questions by subspeciality.

The table shows the number of questions included in each subspeciality category. Values are presented as counts and percentages of the total number of questions (n = ___).
```

自分で書く欄:

```markdown
Table 1. ________________________________________________.

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


### Exercise 7-2: Table 2 の Legend を書く

Table 2 は、各 LLM の全体 accuracy と分野別 accuracy を示す表です。accuracy の定義、単位、略語を含めて書いてください。

```markdown
Table 2. Overall and subspeciality-specific accuracy of large language models.

Accuracy was calculated as the percentage of correctly answered questions. Rows indicate models, and columns indicate overall accuracy and accuracy for each subspeciality. LLM = large language model.
```

自分で書く欄:

```markdown
Table 2. ________________________________________________.

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


## 8) Figure Legend を書く

### Figure Legend とは

Figure Legend（図の説明文）は、図を見た読者が「何を比較しているのか」「軸や色が何を意味するのか」「エラーバーや線が何を示すのか」を理解できるようにする説明です。

Figure Legend に含める内容:
- 図番号と短いタイトル: `Figure 1. Comparison of model accuracy with student performance.` のように書きます。
- 図の内容: 何を、どのグループで、どの指標により比較したかを書きます。
- 軸・色・線・エラーバーの意味: `The dashed line indicates...` のように説明します。
- 略語の説明: LLM、SD などを定義します。

注意点:
- 図から読み取れる主要な傾向を少し書いてもよいですが、詳細な解釈は Results 本文に書きます。
- 図の legend は、読者が本文を読まなくても図を理解できる程度に具体的にします。


### Exercise 8-1: Figure 1 の Legend を書く

Figure 1 は、受験生平均と LLM の正解率を比較した図です。受験生の平均正解率と SD の具体的な数値、破線、帯など、自分の図で使った要素を説明してください。

```markdown
Figure 1. Comparison of LLM accuracy with student performance.

Bars show the overall accuracy of each LLM. The dashed horizontal line indicates the mean score of examinees (__%), and the shaded area indicates one standard deviation (SD, __%). LLM = large language model; SD = standard deviation.
```

自分で書く欄:

```markdown
Figure 1. ________________________________________________.

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


### Exercise 8-2: Figure 2 の Legend を書く

Figure 2 は、分野別 accuracy をレーダーチャートで示した図です。各軸、線、塗りつぶし、単位を説明してください。

```markdown
Figure 2. Subspeciality-specific accuracy of LLMs.

The radar chart shows the accuracy of each LLM across subspeciality categories. Each axis represents one subspeciality, and values are shown as percentages. LLM = large language model.
```

自分で書く欄:

```markdown
Figure 2. ________________________________________________.

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


## 9) Results を書く

### Results とは

Results は「データから何が得られたか」を客観的に書くセクションです。ここでは、方法の詳しい説明や、なぜその結果になったかという深い考察は書きません。それらは Materials and Methods や Discussion に分けます。

Results に書く内容:
- 解析に含まれたデータの要約: 最終的に解析に含まれた問題数、分野数、モデル数を短く確認します。詳しいデータの説明や選択基準は Materials and Methods に書きます。
- 表や図で示した主要な数値: 全体 accuracy、分野別 accuracy、受験生平均との比較など、読者に伝えるべき代表的な数値を書きます。
- データから直接言える傾向: どのモデルが高い/低い、どの分野で差が大きい、受験生平均と比べてどうだったかを書きます。
- 表・図への参照: 本文中で `Table 1`, `Table 2`, `Figure 1`, `Figure 2` を必ず引用します。

表や図を引用するときの考え方:
- `Table 1` は、解析に含まれた問題の分野別内訳を示すときに引用します。
- `Table 2` は、モデルごとの全体 accuracy や分野別 accuracy の数値を示すときに引用します。
- `Figure 1` は、LLM と受験生平均を視覚的に比較するときに引用します。
- `Figure 2` は、分野ごとの性能差やモデル間のパターンを説明するときに引用します。
- 本文では、表や図のすべての数値を繰り返すのではなく、最も重要な結果だけを選んで書きます。

避けること:
- 「なぜそうなったか」の推測を長く書く。
- 表や図の数値をすべて文章で繰り返す。
- Table や Figure を置くだけで、本文中で引用しない。
- 主観的な表現を使う。例: 「すごく良い」「かなり悪い」など。


### Exercise 9: Results のドラフトを書く

以下のヒントをもとに、自分が作成した表・図の数値に合わせて Results を英語で書いてください。ここでは英文の完成例は示しません。数値は必ず自分の解析結果から入力してください。

書くときのヒント:
- 1段落目: 最終的に解析に含まれた問題数、分野数、モデル数を短く書く。分野別の問題数は `Table 1` を引用する。
- 2段落目: 全体 accuracy の結果を書く。最も高かったモデル、次に高かったモデル、最も低かったモデルを選び、`Table 2` を引用する。
- 3段落目: 受験生平均との比較を書く。受験生平均より高い/低い/同程度のモデルを述べ、`Figure 1` を引用する。
- 4段落目: 分野別 accuracy の結果を書く。分野によって性能差があったか、どの分野で高い/低い傾向があったかを述べ、`Table 2` と `Figure 2` を引用する。
- 各段落では、表や図の全数値を繰り返さず、本文で強調すべき主要な数値だけを選ぶ。
- Results では、なぜその差が出たかという解釈は詳しく書かない。解釈は Discussion に回す。

自分で書く欄:

```markdown
# Results

__________________________________________________________
__________________________________________________________
__________________________________________________________

__________________________________________________________
__________________________________________________________
__________________________________________________________

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


## 10) Materials and Methods を書く

### Materials and Methods とは

Materials and Methods は「誰かが同じ解析を再現できるように、何を使って、どのように解析したか」を書くセクションです。Results とは異なり、ここでは結果の数値や解釈ではなく、データ、前処理、評価指標、統計・可視化の手順を書きます。

Materials and Methods に書く内容:
- 解析対象の概要: 問題数、分野数、モデル数、解析対象に含めたデータの範囲を書きます。
- データ: どの CSV を使ったか、どの列を使ったか、正解ラベルとモデル回答がどのように保存されていたかを書きます。
- 対象モデル: 評価した LLM 名、比較対象としての student baseline の扱いを書きます。
- 前処理: 欠損値、空白、列名、正解フラグの確認など。
- 評価指標: accuracy の定義、全体 accuracy と分野別 accuracy の計算方法。
- 可視化: 表、棒グラフ、レーダーチャートを作成した方法と使用ライブラリ。

書き方のポイント:
- 過去形で書くことが多いです。例: `Accuracy was calculated...`
- 再現に必要な情報を優先します。
- 結果の良し悪しの解釈は書きません。


### Exercise 10: Materials and Methods のドラフトを書く

以下のヒントをもとに、自分の解析内容に合わせて Materials and Methods を英語で書いてください。ここでは英文の完成例は示しません。

書くときのヒント:
- `Dataset`: 使用した CSV ファイル、解析対象とした問題数、分野数、使用した主な列を書く。正解、モデル回答、正解/不正解フラグが含まれていることを説明する。
- `Models`: 評価したモデル名を列挙する。モデル名の列と `*_ans` の列の違いも必要に応じて説明する。
- `Preprocessing`: 欠損値、余分な空白、列名、データ型、正解フラグを確認したことを書く。
- `Evaluation of model accuracy`: accuracy の定義を書く。全体 accuracy と `Subspeciality` ごとの accuracy をどのように計算したかを書く。
- `Student baseline`: 学生データまたは仮想的な student baseline をどのように扱ったかを書く。平均と標準偏差を計算した場合は、その方法を書く。
- `Tables and figures`: Table 1、Table 2、Figure 1、Figure 2 をどのような目的で作成したかを書く。作成した表・図を `./save_figures` に保存したこと、使用した Python ライブラリも書く。
- Methods では結果の数値や「どのモデルが良かったか」という解釈は書かない。

自分で書く欄:

```markdown
# Materials and Methods

## Dataset
__________________________________________________________
__________________________________________________________
__________________________________________________________

## Evaluation of model accuracy
__________________________________________________________
__________________________________________________________
__________________________________________________________

## Student baseline
__________________________________________________________
__________________________________________________________
__________________________________________________________

## Tables and figures
__________________________________________________________
__________________________________________________________
__________________________________________________________
```


## 11) 提出前チェックリスト

提出前に、以下を確認してください。

- Table 1 と Table 2 に、それぞれ Table Legend がある。
- Table 1、Table 2、Figure 1、Figure 2 が `./save_figures` に保存されている。
- 論文ドラフトの Markdown で、`./save_figures/Table_1.png` などの相対パスを使って自分の表・図にリンクしている。
- Figure 1 と Figure 2 に、それぞれ Figure Legend がある。
- すべての legend で略語を説明している。
- Results では、Table 1、Table 2、Figure 1、Figure 2 を本文中で参照している。
- Results には主要な数値が入っている。
- Materials and Methods には、データ、前処理、accuracy の定義、分野別解析、可視化方法が書かれている。
- Results に方法の詳細を書きすぎていない。
- Materials and Methods に結果の解釈を書いていない。
